Imports

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from pysyncon import Dataprep, Synth
import numpy as np


SCM Model Function

In [ ]:
"""
- csv_path (String): csv dataset
- treatment_countries (list): countries in treatment group
- policy_year (int): year of policy intervention
- title_prefix (String): name of experiment
- donor_countries (list): countries in donor group (if empty, it will use all non-treatment countries from the csv data)
"""
def run_dengue_scm(csv_path, treatment_countries, policy_year, title_prefix, donor_countries, save_filename):
    """
    Runs a Synthetic Control Method experiment and outputs the weights, treatment effects, confidence intervals, and a visual plot.
    """

    df = pd.read_csv(csv_path)
    cols = ['Main Source', '2025']
    df = df.drop(columns=[c for c in cols if c in df.columns])

    df_long = pd.melt(df, id_vars=['Country'], var_name='year', value_name='cases')
    df_long['year'] = df_long['year'].astype(int)
    df_long['cases'] = pd.to_numeric(df_long['cases'], errors='coerce').fillna(0)

    # Post-treatment data cutoff is 2022 due to erratic spike in cases after 2022
    df_long = df_long[df_long['year'] <= 2022]

    treated_data = df_long[df_long['Country'].isin(treatment_countries)].groupby('year', as_index=False)['cases'].mean() # takes the average number of cases if there are multiple countries
    treated_data['Country'] = 'Treatment Group'

    if donor_countries == []: # if an empty list is passed in, all of the non-treatment countries will be used as donors
        donor_data = df_long[~df_long['Country'].isin(treatment_countries)].copy() 
    else:
        valid_donors = [c for c in donor_countries if c not in treatment_countries]
        donor_data = df_long[df_long['Country'].isin(valid_donors)].copy()

    combined_data = pd.concat([treated_data, donor_data], ignore_index=True)
    donor_countries = donor_data['Country'].unique().tolist()

    # timeline bounds
    min_year = combined_data['year'].min()
    max_year = combined_data['year'].max()

    dataprep = Dataprep(
        foo=combined_data,
        predictors=["cases"],
        predictors_op="mean",
        time_predictors_prior=range(min_year, policy_year),
        dependent="cases",
        unit_variable="Country",
        time_variable="year",
        treatment_identifier="Treatment Group",
        controls_identifier=donor_countries,
        time_optimize_ssr=range(min_year, policy_year),
    )

    synth = Synth()
    synth.fit(dataprep=dataprep)

    print(f"\n{'='*50}")
    print(f"EXPERIMENT: {title_prefix}")
    print(f"Policy Year: {policy_year}")
    print(f"Treated Countries: {', '.join(treatment_countries)}")
    print(f"{'='*50}")
    
    print("\n--- DONOR COUNTRY WEIGHTS ---")
    weights = synth.weights()
    active_donors = weights[weights > 0.001].sort_values(ascending=False)
    print(active_donors)

    print("\n--- POLICY EFFECTIVENESS (Post-Treatment Gap) ---")
    post_treatment_years = list(range(policy_year, max_year + 1))
    treatment_effect = synth.att(time_period=post_treatment_years)
    print(treatment_effect)

    # Plot the paths
    # synth.path_plot(time_period=range(min_year, max_year + 1), treatment_time=policy_year) # this is the base plotting method through the synth library

    plt.figure(figsize=(10, 6))

    actual_data = combined_data[combined_data['Country'] == 'Treatment Group'].sort_values('year')
    time_periods = actual_data['year'].values
    actual_cases = actual_data['cases'].values

    weights = synth.weights()
    synthetic_cases = [0] * len(time_periods)

    for country, weight in weights.items():
        if weight > 0: 
            country_data = combined_data[combined_data['Country'] == country].sort_values('year')['cases'].values
            synthetic_cases = [sync + (weight * real) for sync, real in zip(synthetic_cases, country_data)]

    plt.plot(time_periods, actual_cases, color='black', linewidth=2, label="Actual Treatment Group")
    plt.plot(time_periods, synthetic_cases, color='blue', linestyle='--', linewidth=2, label="Synthetic Control")

    plt.axvline(x=policy_year, color='red', linestyle=':', linewidth=2, label='Policy Implemented')
    plt.title(f'Synthetic Control: {title_prefix}')
    plt.ylabel('Average Dengue Cases')
    plt.xlabel('Year')
    plt.legend()
    plt.grid(True, alpha=0.3)

    actual_arr = np.array(actual_cases)
    synthetic_arr = np.array(synthetic_cases)
    time_arr = np.array(time_periods)

    pre_treat_mask = time_arr < policy_year
    actual_pre = actual_arr[pre_treat_mask]
    synthetic_pre = synthetic_arr[pre_treat_mask]

    l2_normalized = np.linalg.norm(actual_pre - synthetic_pre)
    rmse = np.sqrt(np.mean((actual_pre - synthetic_pre)**2))

    print("\nPRE-TREATMENT FIT METRICS")
    print(f"L2 Norm: {l2_normalized: .2f}")
    print(f"RMSE: {rmse: .2f}")

    plt.savefig(save_filename, dpi=300, bbox_inches='tight')
    print(f"\nSuccessfully saved plot as: {save_filename}")
    # plt.show()

    # Policy Evaluation Methods
    print("\n--- STATISTICAL SUMMARY ---")
    print(synth.summary())
    
    print("\n--- CONFIDENCE INTERVALS ---") # Warning: runs slowly and takes time to display the CIs because of the low tolerance
    try:
        ci_df = synth.confidence_interval(
            alpha=0.05,
            time_periods=post_treatment_years,
            tol=0.05,        
            max_iter=1000,   # max iterations increased to 1000 which increases runtime
            verbose=False
        )

        print(ci_df)
    except Exception as e:
        print(f"Confidence intervals still failed to converge: {e}")
 

CSV Reading

In [ ]:
csv_file = "cleaned_global_dengue_data.csv"
lac_countries = [
    'ANTIGUA AND BARBUDA', 'ARGENTINA', 'ARUBA', 'BARBADOS', 'BELIZE', 
    'BOLIVIA', 'BRAZIL', 'CAYMAN ISLANDS', 'CHILE', 'COLOMBIA', 'COSTA RICA', 
    'CUBA', 'DOMINICA', 'DOMINICAN REPUBLIC', 'ECUADOR', 'EL SALVADOR', 
    'FRENCH GUIANA', 'GRENADA', 'GUADELOUPE', 'GUATEMALA', 'GUYANA', 'HONDURAS', 
    'JAMAICA', 'MARTINIQUE', 'MEXICO', 'MONTSERRAT', 'NICARAGUA', 'PANAMA', 
    'PARAGUAY', 'PERU', 'PUERTO RICO', 'SAINT BARTHELEMY', 'SAINT KITTS AND NEVIS', 
    'SAINT LUCIA', 'SAINT MARTIN', 'SAINT VINCENT AND THE GRENADINES', 'SURINAME', 
    'TRINIDAD AND TOBAGO', 'TURKS AND CAICOS ISLANDS', 'URUGUAY', 'VENEZUELA', 
    'VIRGIN ISLANDS (UK)', 'VIRGIN ISLANDS (US)'
]

Functions Calls

In [ ]:
# 2020 Policy Countries
run_dengue_scm(csv_file, ['GUATEMALA'], 2020, "Guatemala", [], 'guatemala')
run_dengue_scm(csv_file, ['TRINIDAD AND TOBAGO'], 2020, "T&T", lac_countries, 'T&T')

# 2021 Policy Countries
run_dengue_scm(csv_file, ['PERU'], 2021, "Peru", lac_countries, 'peru')

# 2022 Policy Countries
run_dengue_scm(csv_file, ['HONDURAS'], 2022, "Honduras", lac_countries, 'honduras')